# Steering Validation

This notebook verifies that `ExpertSteerer` is mechanistically working as intended — not just that model outputs differ, but that the routing hooks are actually firing correctly at the gate level.

## Candidates

The experts validated here are the **Stage 1 candidates**: the (layer, expert) pairs identified by Stage 1's Risk Difference profiling as most strongly associated with a given behaviour. Specifically, an expert qualifies as a candidate only if it ranks in the top-`CANDIDATE_N` by |RD| on **both** the frequency metric and the logit metric independently for its layer — the dual-metric intersection filter from `stage1/analysis/03_candidate_selection.ipynb`. These are the exact experts suppressed in Stage 2, so validating them here confirms the intervention is mechanistically sound before interpreting behavioural results.

## Conditions

Data is produced by running `validate_steering.py` on the cluster, which passes N=200 prompts through the model under three conditions per axis:

- **Baseline** — no intervention, natural routing
- **Hard** — targeted experts are completely blocked: the hook intercepts the gate output and replaces any suppressed expert with the highest-scoring non-suppressed alternative
- **Soft** — targeted experts have their gate logit shifted down by `strength × RD score` before routing, via a precomputed hidden-state perturbation `δh`

## Prompt Format

The prompts used for each axis **exactly match the experimental condition** in which the target experts were identified — because expert routing rates are input-dependent, and we need to measure suppression under the same activation regime as the actual experiment:

| Axis | Direction | Prompt format | Mirrors task |
|---|---|---|---|
| Safety | Negative (suppress compliance) | AdvBench harmful instruction → chat template → forced prefix `"Sure! Here is..."` appended | `safety_safe` |
| Safety | Positive (suppress refusal) | AdvBench harmful instruction wrapped with safety system prompt | `safety_unsafe` |
| Faithfulness | Negative (suppress confabulation) | FaithEval-Counterfactual record: `Context: ... Question: ... Options: ...` | `faith_cf` |

Only the `negative` direction is validated for faithfulness. Both directions are validated for safety.

In [1]:
import json
import os
import pandas as pd

RESULT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'stage2', 'validation_result.json')
if not os.path.exists(RESULT_PATH):
    RESULT_PATH = '../validation_result.json'

with open(RESULT_PATH) as f:
    data = json.load(f)

cfg = data['config']
print(f"candidate_n={cfg['candidate_n']}  soft_strength={cfg['soft_strength']}  n={cfg['n']}")

candidate_n=3  soft_strength=0.5  n=200


## Column Definitions

Each row is one targeted (layer, expert) pair. Expert indices are **per-layer** — expert 35 at layer 2 and expert 35 at layer 18 are completely different components; the index is just a slot number within that layer's expert pool.

| Column | What it means |
|---|---|
| **Baseline Rate** | Fraction of top-k routing slots won by this expert across all probe tokens, with no intervention. E.g. `0.044` means it was selected in 4.4% of all token-layer routing decisions. |
| **Hard Rate** | Same fraction under hard deactivation. Must be `0.000` — the hook replaces this expert every time it would have been selected. |
| **Hard OK** | `True` if hard rate is exactly zero. This is the primary mechanistic check. |
| **Soft Rate** | Same fraction under soft suppression. Should be lower than baseline — the expert's gate logit is shifted down, so it wins less often, but can still win if it's sufficiently dominant. |
| **Soft Rate Reduced** | `True` if soft rate < baseline rate. |
| **Expected Logit Shift** | The logit shift we asked for: `soft_strength × RD score` for this expert. Negative means we're pushing the logit down. |
| **Actual Logit Shift** | The logit shift measured directly from the gate inputs during the soft run. Should closely match expected — if it does, the hidden-state perturbation (`δh`) is landing correctly. |

In [2]:
def build_df(axis_data):
    rows = []
    for layer_str, layer_val in axis_data['layers'].items():
        for expert_str, e in layer_val['experts'].items():
            rows.append({
                'Layer':                int(layer_str),
                'Expert':               int(expert_str),
                'Baseline Rate':        e['baseline_rate'],
                'Hard Rate':            e['hard_rate'],
                'Hard OK':              e['hard_ok'],
                'Soft Rate':            e['soft_rate'],
                'Soft Rate Reduced':    e['soft_rate_reduced'],
                'Expected Logit Shift': e['expected_logit_shift'],
                'Actual Logit Shift':   e['actual_logit_shift'],
            })
    return pd.DataFrame(rows).sort_values(['Layer', 'Expert']).reset_index(drop=True)


def style_df(df):
    def colour_hard_rate(val):
        return 'background-color: #c8f7c5; font-weight: bold' if val == 0.0 else 'background-color: #f7c5c5; font-weight: bold'

    def colour_bool(val):
        return 'color: green; font-weight: bold' if val else 'color: red; font-weight: bold'

    def colour_shift_diff(row):
        styles = [''] * len(row)
        exp = row['Expected Logit Shift']
        act = row['Actual Logit Shift']
        if exp is not None and exp != 0:
            rel_err = abs((act - exp) / exp)
            col = '#c8f7c5' if rel_err < 0.15 else '#fff3cd' if rel_err < 0.40 else '#f7c5c5'
            styles[list(row.index).index('Actual Logit Shift')] = f'background-color: {col}'
        return styles

    styler = (
        df.style
        .map(colour_hard_rate, subset=['Hard Rate'])
        .map(colour_bool,      subset=['Hard OK', 'Soft Rate Reduced'])
        .apply(colour_shift_diff,   axis=1)
        .format({
            'Baseline Rate':        '{:.4f}',
            'Hard Rate':            '{:.4f}',
            'Soft Rate':            '{:.4f}',
            'Expected Logit Shift': '{:+.4f}',
            'Actual Logit Shift':   '{:+.4f}',
        })
    )

    if hasattr(styler, 'hide'):
        return styler.hide(axis='index')
    return styler.hide_index()

## Safety — Negative Direction (suppress compliance experts)

These are the Stage 1 candidates most associated with unsafe compliance — experts that fire preferentially when the model is producing harmful output. Validated on AdvBench prompts with forced prefix (39 candidates across 24 layers).

In [3]:
safety_neg_df = build_df(data['safety_negative'])
style_df(safety_neg_df)

Layer,Expert,Baseline Rate,Hard Rate,Hard OK,Soft Rate,Soft Rate Reduced,Expected Logit Shift,Actual Logit Shift
2,35,0.0373,0.0000,True,0.0049,True,-1.0436,-1.0796
2,63,0.0433,0.0000,True,0.0135,True,-1.1104,-0.9193
3,22,0.0431,0.0000,True,0.0246,True,-1.0694,-0.9907
4,18,0.0270,0.0000,True,0.0111,True,-0.7034,-0.7482
4,32,0.0036,0.0000,True,0.0005,True,-0.8573,-0.7790
5,32,0.0079,0.0000,True,0.0000,True,-0.9399,-0.9087
5,61,0.0131,0.0000,True,0.0007,True,-0.8058,-0.8213
7,8,0.0461,0.0000,True,0.0161,True,-0.8057,-0.7091
8,43,0.0071,0.0000,True,0.0000,True,-0.9955,-0.9502
9,27,0.0193,0.0000,True,0.0003,True,-0.7272,-0.7955


### What this validation measures — and what it does not

**What the validation ingests.** Each forward pass in this validation processes only the input sequence: the AdvBench prompt plus the forced prefix (for the negative direction) or the safety system prompt (for the positive direction). No output tokens are present. The model performs a single prefill pass and stops.

**What the actual experiment involves.** In Stage 2, `model.generate` is called with `use_cache=False`. This means that at each generation step `t`, the model performs a full forward pass over the entire sequence constructed so far: `[prompt + prefix + token_1 + ... + token_t]`. By step 50 of a harmful completion, the hidden states at every layer are conditioned on 50 tokens of generated output. Those hidden states are fundamentally different in character from the hidden states produced during prefill of the prompt alone — they encode the model actively mid-generation, not reading an instruction.

This validation therefore does not observe the hidden states that occur during generation. That is a deliberate and justified choice, for the following reasons.

**Hard deactivation is h-independent.** The hard hook operates on the gate's output tensor — it inspects the selected expert indices and substitutes any suppressed expert with the highest-scoring non-suppressed alternative. At no point does the hook read or depend on the hidden state `h`. The mechanism is therefore identical whether `h` is a prefill token, a generation token at step 1, or a generation token at step 100. A hard rate of `0.000` across 200 prompts during prefill constitutes a complete proof that the hook is functioning correctly; running the same check during generation would activate the same conditional logic and produce the same result.

**The soft logit shift is h-independent.** The soft intervention adds a fixed perturbation vector `δh` to the hidden state before the gate runs. The resulting change in the gate logit for expert `i` is:

&emsp; `Δlogit_i = F.linear(δh, W)[i] = δh · W[i]`

This is a dot product between `δh` and the `i`-th row of the gate weight matrix `W`. It depends only on `δh` and `W` — not on `h`. Regardless of what sequence the model is processing, adding `δh` to the hidden state shifts expert `i`'s logit by the same fixed amount. The *Expected* and *Actual Logit Shift* columns above are therefore valid for every forward pass in a generation run, not just for prefill tokens.

**What generation-based validation would add.** The one quantity that prefill validation does not capture is the *baseline routing rate during generation* — how frequently each expert is selected while the model is producing output tokens. During generation, hidden states reflect the growing output context and may activate different experts at different rates than during prefill. A generation-based validation would yield more ecologically grounded routing rate estimates. However, since the correctness of both the hard hook and the soft logit shift is provably independent of `h`, those routing rates carry no additional information about whether the steering mechanism is functioning correctly. They would describe *how active* the target experts are mid-generation, but not *whether* they are being suppressed — and it is suppression correctness that this validation is designed to establish.

## Safety — Positive Direction (suppress refusal experts)

These are the Stage 1 candidates most associated with safe refusal — experts that fire preferentially when the model is declining a harmful request. Validated on AdvBench prompts with safety system prompt (46 candidates across 24 layers).

In [4]:
safety_pos_df = build_df(data['safety_positive'])
style_df(safety_pos_df)

Layer,Expert,Baseline Rate,Hard Rate,Hard OK,Soft Rate,Soft Rate Reduced,Expected Logit Shift,Actual Logit Shift
1,8,0.0294,0.0000,True,0.0021,True,-1.3086,-1.3086
3,35,0.0272,0.0000,True,0.0056,True,-0.9900,-0.9699
3,63,0.0280,0.0000,True,0.0055,True,-1.1060,-1.1188
4,12,0.0213,0.0000,True,0.0043,True,-1.4515,-1.4658
4,25,0.0242,0.0000,True,0.0000,True,-1.6417,-1.6854
5,31,0.0060,0.0000,True,0.0000,True,-1.4592,-1.4787
5,49,0.0656,0.0000,True,0.0104,True,-1.2499,-1.3470
5,57,0.0378,0.0000,True,0.0197,True,-1.0939,-1.0418
6,52,0.0257,0.0000,True,0.0129,True,-1.0188,-1.0381
6,53,0.0476,0.0000,True,0.0093,True,-1.4341,-1.4897


## Faithfulness — Negative Direction (suppress confabulation experts)

These are the Stage 1 candidates most associated with context-ignoring confabulation — experts that fire preferentially when the model ignores the provided context and generates from parametric memory. Validated on FaithEval-Counterfactual records (28 candidates across 19 layers).

In [5]:
faith_df = build_df(data['faithfulness'])
style_df(faith_df)

Layer,Expert,Baseline Rate,Hard Rate,Hard OK,Soft Rate,Soft Rate Reduced,Expected Logit Shift,Actual Logit Shift
1,6,0.0107,0.0000,True,0.0000,True,-1.5292,-1.5291
1,8,0.0165,0.0000,True,0.0000,True,-1.5036,-1.5036
3,60,0.0094,0.0000,True,0.0001,True,-1.3281,-1.3162
5,7,0.0254,0.0000,True,0.0082,True,-1.0865,-1.0086
5,34,0.0239,0.0000,True,0.0036,True,-1.2345,-1.2233
6,38,0.0289,0.0000,True,0.0047,True,-1.2021,-1.1690
7,62,0.0160,0.0000,True,0.0030,True,-0.8955,-0.9185
8,16,0.0246,0.0000,True,0.0061,True,-0.8469,-0.8837
10,19,0.0055,0.0000,True,0.0000,True,-2.0697,-1.9510
12,13,0.0279,0.0000,True,0.0009,True,-0.9097,-0.8925


### Note on prefill validity for faithfulness

The same h-independence argument from the safety section applies here. Additionally, the faithfulness benchmarks (FaithEval-Counterfactual, MCTest) are multiple-choice tasks: the model reads a context passage and question, then outputs a single letter. The generation phase is therefore minimal — at most a few tokens — meaning the prefill of the context prompt constitutes the overwhelming majority of the computation and the routing signal. Prefill-based validation is therefore not only mechanistically sufficient but also a close approximation of the actual experimental regime.

## Summary

**Hard check** — every targeted expert must have `Hard Rate == 0.000`. If any fail, the output hook is not firing in the correct order relative to the observation hook, meaning the suppression is not taking effect.

**Soft check** — `Soft Rate` should be below `Baseline Rate` for every expert. The logit shift columns confirm the hidden-state perturbation (`δh`) is producing the intended change in gate scores: green = within 15% of expected, yellow = within 40%, red = off by more than 40%.

> **Note on logit shift discrepancies.** Minor differences between expected and actual logit shift are expected and benign. Two sources contribute:
>
> 1. **Float16 precision loss.** `δh` is computed analytically in float32 but must be cast to float16 to match the model's inference dtype before being added to the hidden state. float16 has limited precision (~3 significant decimal digits), so the applied shift is a slightly rounded version of what was requested.
>
> 2. **Matrix inversion error amplified by shift magnitude.** `δh` is derived by inverting `WW^T` — the product of the gate's weight matrix with itself. This inversion is numerically approximate in floating point, and the resulting error in `δh` scales proportionally with the size of the requested shift (`strength × RD score`). Experts with larger |RD| scores therefore request larger shifts, and the same relative inversion error translates into a larger absolute discrepancy — which is why the yellow cases tend to be the experts with the strongest RD signal, not a random subset.
>
> Neither source indicates incorrect hook behaviour. The hooks are firing as intended; the deviation is purely a numerical artefact of float16 inference and finite-precision matrix inversion.

In [6]:
combined = pd.concat([
    safety_neg_df.assign(Axis='Safety (negative)'),
    safety_pos_df.assign(Axis='Safety (positive)'),
    faith_df.assign(Axis='Faithfulness (negative)'),
])

n_total   = len(combined)
n_hard_ok = combined['Hard OK'].sum()
n_soft_ok = combined['Soft Rate Reduced'].sum()

print(f"Hard check : {n_hard_ok}/{n_total} experts have routing rate == 0.000")
print(f"Soft check : {n_soft_ok}/{n_total} experts show reduced routing rate vs baseline")

if n_hard_ok == n_total:
    print("\nHard steering: ALL PASS")
else:
    print("\nHard steering: FAILURES DETECTED")
    print(combined[~combined['Hard OK']][['Axis', 'Layer', 'Expert', 'Hard Rate']].to_string(index=False))

if n_soft_ok < n_total:
    print("\nSoft steering: experts NOT reduced:")
    cols = ['Axis', 'Layer', 'Expert', 'Baseline Rate', 'Soft Rate']
    print(combined[~combined['Soft Rate Reduced']][cols].to_string(index=False))

Hard check : 113/113 experts have routing rate == 0.000
Soft check : 113/113 experts show reduced routing rate vs baseline

Hard steering: ALL PASS
